# 12 — Supervised Fine-Tuning (SFT)

**Network LLM Engineering — Part III — Adaptation**

### Learning goals
- Understand prompt/completion loss
- Run a teaching-scale SFT configuration
- Measure behavior, not only train loss

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## SFT

SFT updates a pretrained model so target completions become more probable for given prompts.
For a network assistant, SFT can teach:
- output schema,
- terminology,
- incident taxonomy,
- verification-first troubleshooting,
- concise explanation level.

It should not be used as the primary storage for dynamic topology.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

train = load_dataset("json", data_files=str(DATA/"network_sft_train.jsonl"), split="train")
print(train)
print(train[0]["prompt"][-1]["content"])
print(train[0]["completion"][0]["content"])

## Minimal trainer pattern

The next cell **constructs** a current TRL trainer configuration. Run the actual training after Notebook 13 adds LoRA,
because full SFT is unnecessarily expensive for this teaching project.

In [ ]:
from trl import SFTConfig
cfg = SFTConfig(
    output_dir="./sft-demo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_length=512,
    completion_only_loss=True,
    report_to="none",
)
print(cfg)

## Training loss is necessary, not sufficient

Falling loss means the model is better at predicting the training targets.
It does not prove:
- factual correctness,
- generalization,
- safety,
- robustness to new topologies,
- tool-grounded behavior.

### Exercise

Identify three networking behaviors you would SFT and three facts you would keep in tools/RAG.